# Online learning — prequential evaluation
Pair C notebook. Runs the adaptive River learner vs a static AutoML-winner baseline on a 200-event simulated stream with a planted query-style drift at event 100. Saves `reports/prequential.png` and reports the post-drift delta for the §6.C acceptance claim.

In [1]:
import os
from pathlib import Path
import yaml
import numpy as np

# Notebook runs from notebooks/; chdir to repo root so the default relative
# paths inside load_chunks() / _load_holdout_queries() / load_automl_weight() work.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from csai415.retrieve import HybridRetriever, RetrieverConfig, load_chunks, make_retriever_fn
from csai415.online import run_prequential, plot_prequential

RUNCARD = Path("configs/winning_runcard.yaml")
REPORTS = Path("reports")
REPORTS.mkdir(parents=True, exist_ok=True)

card = yaml.safe_load(RUNCARD.read_text())
best = card["automl"]["best_params"]
print(f"Loaded winning AutoML params: {best}")

Loaded winning AutoML params: {'metric': 'l2', 'svd_dim': None, 'normalize': False, 'hybrid_weight': 0.8102310529569093, 'candidate_k': 24}


## Build retriever with AutoML-winning config
We use the AutoML-winning retriever config so the static baseline `hybrid_weight` (loaded inside `run_prequential` from the runcard) reflects Pair B's best static setup. The learner overrides only `hybrid_weight` per-call; everything else (metric, SVD, normalize, candidate_k) stays at the tuned values.

In [2]:
cfg = RetrieverConfig(
    metric=best["metric"],
    svd_dim=best["svd_dim"],
    normalize=best["normalize"],
    hybrid_weight=best["hybrid_weight"],
    candidate_k=best["candidate_k"],
    seed=42,
)

chunks_df = load_chunks()
retriever = HybridRetriever(chunks_df, cfg)
retriever_fn = make_retriever_fn(retriever)
print(f"Built retriever over {len(chunks_df)} chunks")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Built retriever over 6020 chunks


## Run the prequential loop
200 events, drift planted at event 100 (query-style shift: natural-language claims -> 2-token keyword queries). The learner adapts; the static baseline replays the same stream at a fixed weight; ADWIN watches the static probe.

In [3]:
state = run_prequential(retriever_fn, n_events=200, drift_at=100)
print(f"Stream length: {len(state.prequential_ndcg5)} events")
print(f"ADWIN fired at events: {state.drift_events}")

Stream length: 200 events
ADWIN fired at events: [159]


## Save the figure

In [4]:
out = plot_prequential(state, out_path=REPORTS / "prequential.png")
print(f"Wrote {out}")

Wrote reports\prequential.png


## Post-drift delta — the §6.C acceptance claim
Acceptance bar: "online learner improves NDCG@5 by >= 5% vs static after the drift point."

In [5]:
DRIFT_AT = state.drift_at
adaptive_post = float(np.mean(state.prequential_ndcg5[DRIFT_AT:]))
static_post = float(np.mean(state.baseline_ndcg5[DRIFT_AT:]))
adaptive_pre = float(np.mean(state.prequential_ndcg5[:DRIFT_AT]))
static_pre = float(np.mean(state.baseline_ndcg5[:DRIFT_AT]))

print(f"Pre-drift  : adaptive NDCG@5 = {adaptive_pre:.4f}   static NDCG@5 = {static_pre:.4f}")
print(f"Post-drift : adaptive NDCG@5 = {adaptive_post:.4f}   static NDCG@5 = {static_post:.4f}")
print(f"Post-drift delta (adaptive - static): {adaptive_post - static_post:+.4f}")
passes = (adaptive_post - static_post) >= 0.05
print(f">= 5% improvement claim: {'PASS' if passes else 'MISS'}")

Pre-drift  : adaptive NDCG@5 = 0.5488   static NDCG@5 = 0.6000
Post-drift : adaptive NDCG@5 = 0.3317   static NDCG@5 = 0.3011
Post-drift delta (adaptive - static): +0.0306
>= 5% improvement claim: MISS


In [ ]:
from csai415.online import run_c3
from csai415.retrieve import HybridRetriever

# Build retriever the same way as the rest of your notebook
retriever = HybridRetriever(...)  

run_c3(
    retriever_fn=lambda q, k, w: retriever.retrieve(q, k=k, hybrid_weight=w)
)

In [ ]:
from csai415.online import simulate_feedback_stream, build_drift_detector, _reward
from csai415.online import load_automl_weight
from river import drift as river_drift

stream = simulate_feedback_stream()
static_w = load_automl_weight()

for delta in [0.5, 0.1, 0.01, 0.002, 0.001]:
    detector = river_drift.ADWIN(delta=delta)
    firings = []
    for i, q, relevant in stream:
        # simulate static retriever reward (you'll need your retriever here)
        # for a quick sweep just feed random binary values matching the real hit-rate
        import random
        reward = 1.0 if random.random() < (0.67 if i < 800 else 0.35) else 0.0
        detector.update(reward)
        if detector.drift_detected:
            firings.append(i)
    print(f"delta={delta}: firings at {firings}")